In [0]:
transactions_path = "/Volumes/workspace/default/casinoguard_data/transactions.csv"
accounts_path = "/Volumes/workspace/default/casinoguard_data/accounts.csv"

transactions = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(transactions_path)

accounts = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(accounts_path)


In [0]:
print("Total transactions:", transactions.count())
print("Total accounts:", accounts.count())

display(transactions)
display(accounts)

Total transactions: 49
Total accounts: 4


id,account,from,to,amount,timestamp,type
NORMAL001,PLAYER001,PLAYER001,CASINO,200,2026-11-05T09:00:00.000Z,deposit
NORMAL002,PLAYER001,PLAYER001,CASINO,50,2026-11-05T09:10:00.000Z,wager
NORMAL003,PLAYER001,CASINO,PLAYER001,100,2026-11-05T09:30:00.000Z,withdrawal
CHAIN001,CHAIN_A,CHAIN_A,CHAIN_B,5000,2026-11-05T10:00:00.000Z,transfer
CHAIN002,CHAIN_B,CHAIN_B,CHAIN_C,4900,2026-11-05T10:04:00.000Z,transfer
CHAIN003,CHAIN_C,CHAIN_C,CHAIN_D,4800,2026-11-05T10:08:00.000Z,transfer
STRUCT001,STRUCT_A,STRUCT_A,CAGE001,4000,2026-11-05T11:00:00.000Z,deposit
STRUCT002,STRUCT_A,STRUCT_A,CAGE002,3500,2026-11-05T11:15:00.000Z,deposit
STRUCT003,STRUCT_A,STRUCT_A,CAGE003,3000,2026-11-05T11:30:00.000Z,deposit
CASH001,CASH_A,CASH_A,CASINO,9000,2026-11-05T12:00:00.000Z,deposit


account,phone,email,payment_card
LINK_A,555-1111,linka@example.com,CARD-101
LINK_B,555-1111,linkb@example.com,CARD-101
LINK_C,555-2222,linkc@example.com,CARD-202
NORMAL_ACCOUNT,555-3333,normal@example.com,CARD-303


In [0]:
display(
    transactions.groupBy("type")
    .count()
    .orderBy("type")
)

type,count
deposit,5
transfer,40
wager,2
withdrawal,2


In [0]:
print("Total rows:", transactions.count())
print("Unique transaction IDs:", transactions.select("id").distinct().count())

display(transactions.orderBy("timestamp"))

Total rows: 49
Unique transaction IDs: 49


id,account,from,to,amount,timestamp,type
VOLUME_HIST_0_0,VOLUME_A,VOLUME_A,CASINO,100,2026-10-29T10:00:00.000Z,transfer
VOLUME_HIST_0_1,VOLUME_A,VOLUME_A,CASINO,100,2026-10-29T11:00:00.000Z,transfer
VOLUME_HIST_1_0,VOLUME_A,VOLUME_A,CASINO,100,2026-10-30T10:00:00.000Z,transfer
VOLUME_HIST_1_1,VOLUME_A,VOLUME_A,CASINO,100,2026-10-30T11:00:00.000Z,transfer
VOLUME_HIST_2_0,VOLUME_A,VOLUME_A,CASINO,100,2026-10-31T10:00:00.000Z,transfer
VOLUME_HIST_2_1,VOLUME_A,VOLUME_A,CASINO,100,2026-10-31T11:00:00.000Z,transfer
VOLUME_HIST_3_0,VOLUME_A,VOLUME_A,CASINO,100,2026-11-01T10:00:00.000Z,transfer
VOLUME_HIST_3_1,VOLUME_A,VOLUME_A,CASINO,100,2026-11-01T11:00:00.000Z,transfer
VOLUME_HIST_4_0,VOLUME_A,VOLUME_A,CASINO,100,2026-11-02T10:00:00.000Z,transfer
VOLUME_HIST_4_1,VOLUME_A,VOLUME_A,CASINO,100,2026-11-02T11:00:00.000Z,transfer


In [0]:
transactions.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("transactions_bronze")

accounts.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("accounts_bronze")

In [0]:
%sql
SELECT * FROM transactions_bronze;

id,account,from,to,amount,timestamp,type
NORMAL001,PLAYER001,PLAYER001,CASINO,200,2026-11-05T09:00:00.000Z,deposit
NORMAL002,PLAYER001,PLAYER001,CASINO,50,2026-11-05T09:10:00.000Z,wager
NORMAL003,PLAYER001,CASINO,PLAYER001,100,2026-11-05T09:30:00.000Z,withdrawal
CHAIN001,CHAIN_A,CHAIN_A,CHAIN_B,5000,2026-11-05T10:00:00.000Z,transfer
CHAIN002,CHAIN_B,CHAIN_B,CHAIN_C,4900,2026-11-05T10:04:00.000Z,transfer
CHAIN003,CHAIN_C,CHAIN_C,CHAIN_D,4800,2026-11-05T10:08:00.000Z,transfer
STRUCT001,STRUCT_A,STRUCT_A,CAGE001,4000,2026-11-05T11:00:00.000Z,deposit
STRUCT002,STRUCT_A,STRUCT_A,CAGE002,3500,2026-11-05T11:15:00.000Z,deposit
STRUCT003,STRUCT_A,STRUCT_A,CAGE003,3000,2026-11-05T11:30:00.000Z,deposit
CASH001,CASH_A,CASH_A,CASINO,9000,2026-11-05T12:00:00.000Z,deposit


In [0]:
%sql
SELECT * FROM accounts_bronze;

account,phone,email,payment_card
LINK_A,555-1111,linka@example.com,CARD-101
LINK_B,555-1111,linkb@example.com,CARD-101
LINK_C,555-2222,linkc@example.com,CARD-202
NORMAL_ACCOUNT,555-3333,normal@example.com,CARD-303


In [0]:
from pyspark.sql.functions import col, to_timestamp

transactions_silver = (
    transactions
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("timestamp", to_timestamp(col("timestamp")))
    .filter(col("amount").isNotNull())
    .filter(col("timestamp").isNotNull())
)

In [0]:
transactions_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("transactions_silver")

In [0]:
accounts.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("accounts_silver")

In [0]:
print("Silver transactions:", transactions_silver.count())
print("Silver accounts:", accounts.count())

transactions_silver.printSchema()

Silver transactions: 49
Silver accounts: 4
root
 |-- id: string (nullable = true)
 |-- account: string (nullable = true)
 |-- from: string (nullable = true)
 |-- to: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- type: string (nullable = true)



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Detection parameters
THRESHOLD = 10000
WINDOW_SECONDS = 60 * 60
MIN_TRANSACTIONS = 3

# Casino-owned accounts
HOUSE_ACCOUNTS = ["CASINO"]

# Filter eligible transactions
eligible = (
    transactions_silver
    .filter(~F.col("from").isin(HOUSE_ACCOUNTS))
    .filter(F.col("amount") < THRESHOLD)
    .filter(F.col("type") != "wager")
)

# Convert timestamps to Unix seconds
eligible = eligible.withColumn(
    "event_time",
    F.col("timestamp").cast("long")
)

# Rolling 60-minute window for each sender
rolling_window = (
    Window
    .partitionBy("from")
    .orderBy("event_time")
    .rangeBetween(-WINDOW_SECONDS, 0)
)

# Calculate rolling transaction statistics
structuring_results = (
    eligible
    .withColumn(
        "transaction_count",
        F.count("*").over(rolling_window)
    )
    .withColumn(
        "total_amount",
        F.sum("amount").over(rolling_window)
    )
    .withColumn(
        "transaction_ids",
        F.collect_list("id").over(rolling_window)
    )
    .filter(
        (F.col("total_amount") >= THRESHOLD) &
        (F.col("transaction_count") >= MIN_TRANSACTIONS)
    )
    .select(
        F.lit("transaction_structuring").alias("pattern"),
        F.col("from").alias("account"),
        "transaction_ids",
        "transaction_count",
        "total_amount",
        F.lit(60).alias("window_minutes")
    )
    .dropDuplicates(["account", "transaction_ids"])
)

display(structuring_results)

pattern,account,transaction_ids,transaction_count,total_amount,window_minutes
transaction_structuring,STRUCT_A,"List(STRUCT001, STRUCT002, STRUCT003)",3,10500.0,60


In [0]:
structuring_results.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("structuring_alerts_gold")

In [0]:
display(structuring_results)

pattern,account,transaction_ids,transaction_count,total_amount,window_minutes
transaction_structuring,STRUCT_A,"List(STRUCT001, STRUCT002, STRUCT003)",3,10500.0,60


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Original algorithm parameters
MIN_HOPS = 3
MAX_HOPS = 5
MAX_TIME_MINUTES = 15
AMOUNT_SIMILARITY = 0.80

# Match this list to detectors.constants.is_house_account()
HOUSE_ACCOUNTS = ["CASINO"]

# Prepare transaction edges
edges = (
    transactions_silver
    .filter(~F.col("from").isin(HOUSE_ACCOUNTS))
    .filter(~F.col("to").isin(HOUSE_ACCOUNTS))
    .select(
        F.col("from").alias("src"),
        F.col("to").alias("dst"),
        F.col("amount").alias("amount"),
        F.col("timestamp").alias("event_time"),
        F.col("id").alias("transaction_id")
    )
    .filter(F.col("amount").isNotNull())
    .filter(F.col("event_time").isNotNull())
)

# Initialize paths using every transaction as a starting edge
paths = edges.select(
    F.concat(F.col("transaction_id")).alias("path_key"),
    F.array("src", "dst").alias("accounts"),
    F.array("transaction_id").alias("transaction_ids"),
    F.col("dst").alias("current_account"),
    F.col("amount").alias("total_amount"),
    F.col("amount").alias("min_amount"),
    F.col("amount").alias("max_amount"),
    F.col("event_time").alias("start_time"),
    F.col("event_time").alias("last_time"),
    F.lit(1).alias("hops")
)

findings = []

# Extend paths one hop at a time
for target_hops in range(2, MAX_HOPS + 1):

    p = paths.alias("p")
    e = edges.alias("e")

    candidates = (
        p.join(
            e,
            F.col("p.current_account") == F.col("e.src")
        )
        .filter(
            ~F.array_contains(
                F.col("p.accounts"),
                F.col("e.dst")
            )
        )
        .filter(
            F.col("e.event_time") >= F.col("p.last_time")
        )
        .filter(
            (
                F.col("e.event_time").cast("long") -
                F.col("p.start_time").cast("long")
            ) <= MAX_TIME_MINUTES * 60
        )
        .filter(
            (
                F.least(
                    F.col("p.min_amount"),
                    F.col("e.amount")
                ) /
                F.greatest(
                    F.col("p.max_amount"),
                    F.col("e.amount")
                )
            ) >= AMOUNT_SIMILARITY
        )
        .select(
            F.col("p.path_key").alias("path_key"),
            F.col("p.accounts").alias("accounts"),
            F.col("p.transaction_ids").alias("transaction_ids"),
            F.col("p.current_account").alias("current_account"),
            F.col("p.total_amount").alias("total_amount"),
            F.col("p.min_amount").alias("min_amount"),
            F.col("p.max_amount").alias("max_amount"),
            F.col("p.start_time").alias("start_time"),
            F.col("p.last_time").alias("last_time"),
            F.col("e.dst").alias("dst"),
            F.col("e.amount").alias("next_amount"),
            F.col("e.event_time").alias("next_time"),
            F.col("e.transaction_id").alias("next_id")
        )
    )

    # Match original next_hop(): earliest qualifying transfer
    # for each path and neighboring account.
    candidate_window = (
        Window
        .partitionBy("path_key", "dst")
        .orderBy("next_time")
    )

    candidates = (
        candidates
        .withColumn(
            "candidate_rank",
            F.row_number().over(candidate_window)
        )
        .filter(F.col("candidate_rank") == 1)
        .drop("candidate_rank")
    )

    # Build extended paths
    paths = candidates.select(
        F.concat_ws(
            "|",
            F.concat(
                F.col("transaction_ids"),
                F.array("next_id")
            )
        ).alias("path_key"),

        F.concat(
            F.col("accounts"),
            F.array("dst")
        ).alias("accounts"),

        F.concat(
            F.col("transaction_ids"),
            F.array("next_id")
        ).alias("transaction_ids"),

        F.col("dst").alias("current_account"),

        (
            F.col("total_amount") +
            F.col("next_amount")
        ).alias("total_amount"),

        F.least(
            F.col("min_amount"),
            F.col("next_amount")
        ).alias("min_amount"),

        F.greatest(
            F.col("max_amount"),
            F.col("next_amount")
        ).alias("max_amount"),

        F.col("start_time"),
        F.col("next_time").alias("last_time"),

        F.lit(target_hops).alias("hops")
    )

    # Save findings once the minimum hop count is reached
    if target_hops >= MIN_HOPS:
        findings.append(
            paths.select(
                F.lit("suspicious_transaction_chain").alias("pattern"),
                "accounts",
                "transaction_ids",
                "total_amount",
                (
                    F.col("last_time").cast("long") -
                    F.col("start_time").cast("long")
                ).cast("double").__truediv__(60).alias("elapsed_minutes"),
                F.round(
                    F.col("min_amount") / F.col("max_amount"),
                    2
                ).alias("amount_similarity"),
                "hops"
            )
        )

# Combine all detected chains
if findings:
    suspicious_chains = findings[0]

    for result in findings[1:]:
        suspicious_chains = suspicious_chains.unionByName(result)
else:
    suspicious_chains = spark.createDataFrame(
        [],
        """
        pattern STRING,
        accounts ARRAY<STRING>,
        transaction_ids ARRAY<STRING>,
        total_amount DOUBLE,
        elapsed_minutes DOUBLE,
        amount_similarity DOUBLE,
        hops INT
        """
    )

display(suspicious_chains)

pattern,accounts,transaction_ids,total_amount,elapsed_minutes,amount_similarity,hops
suspicious_transaction_chain,"List(CHAIN_A, CHAIN_B, CHAIN_C, CHAIN_D)","List(CHAIN001, CHAIN002, CHAIN003)",14700.0,8.0,0.96,3


In [0]:
suspicious_chains.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("transaction_chains_gold")

In [0]:
from pyspark.sql import functions as F

# Original algorithm parameters
MIN_CASHOUT_RATIO = 0.80
MAX_WAGER_RATIO = 0.20
MAX_TIME_MINUTES = 60

deposits = (
    transactions_silver
    .filter(F.col("type") == "deposit")
    .select(
        F.col("account").alias("account"),
        F.col("id").alias("deposit_id"),
        F.col("amount").alias("deposit_amount"),
        F.col("timestamp").alias("deposit_time")
    )
    .alias("d")
)

subsequent_transactions = (
    transactions_silver
    .select(
        "account",
        "id",
        "amount",
        "type",
        "timestamp"
    )
    .alias("t")
)

cashout_candidates = (
    deposits
    .join(
        subsequent_transactions,
        (F.col("d.account") == F.col("t.account")) &
        (F.col("t.timestamp") > F.col("d.deposit_time")) &
        (
            F.col("t.timestamp") <=
            F.col("d.deposit_time") +
            F.expr(f"INTERVAL {MAX_TIME_MINUTES} MINUTES")
        ),
        "inner"
    )
)

rapid_cashout_results = (
    cashout_candidates
    .groupBy(
        F.col("d.account").alias("account"),
        "deposit_id",
        "deposit_amount",
        "deposit_time"
    )
    .agg(
        F.sum(
            F.when(F.col("t.type") == "withdrawal", F.col("t.amount"))
            .otherwise(0)
        ).alias("total_withdrawn"),

        F.sum(
            F.when(F.col("t.type") == "wager", F.col("t.amount"))
            .otherwise(0)
        ).alias("total_wagered"),

        F.collect_list(
            F.when(
                F.col("t.type") == "withdrawal",
                F.col("t.id")
            )
        ).alias("withdrawal_ids"),

        F.min(
            F.when(
                F.col("t.type") == "withdrawal",
                F.col("t.timestamp")
            )
        ).alias("first_withdrawal_time"),

        F.sum(
            F.when(F.col("t.type") == "withdrawal", 1)
            .otherwise(0)
        ).alias("withdrawal_count")
    )
    .filter(F.col("withdrawal_count") > 0)
    .withColumn(
        "cashout_ratio",
        F.col("total_withdrawn") / F.col("deposit_amount")
    )
    .withColumn(
        "wager_ratio",
        F.col("total_wagered") / F.col("deposit_amount")
    )
    .filter(
        (F.col("cashout_ratio") >= MIN_CASHOUT_RATIO) &
        (F.col("wager_ratio") <= MAX_WAGER_RATIO)
    )
    .select(
        F.lit("rapid_cashout").alias("pattern"),
        "account",
        "deposit_id",
        "withdrawal_ids",
        "deposit_amount",
        "total_withdrawn",
        "total_wagered",
        F.round("cashout_ratio", 4).alias("cashout_ratio"),
        F.round("wager_ratio", 4).alias("wager_ratio"),
        F.round(
            (
                F.col("first_withdrawal_time").cast("long") -
                F.col("deposit_time").cast("long")
            ) / 60,
            2
        ).alias("elapsed_minutes")
    )
)

display(rapid_cashout_results)

pattern,account,deposit_id,withdrawal_ids,deposit_amount,total_withdrawn,total_wagered,cashout_ratio,wager_ratio,elapsed_minutes
rapid_cashout,CASH_A,CASH001,List(CASH003),9000.0,8900.0,100.0,0.9889,0.0111,15.0


In [0]:
rapid_cashout_results.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("rapid_cashouts_gold")

In [0]:
from pyspark.sql import functions as F

transactions_silver = (
    spark.table("transactions_bronze")
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .filter(F.col("amount").isNotNull())
    .filter(F.col("timestamp").isNotNull())
)

transactions_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("transactions_silver")

In [0]:
from pyspark.sql import functions as F

MIN_ACCOUNTS = 3
MAX_ACCOUNTS = 6
HOUSE_ACCOUNTS = ["CASINO"]

edges = (
    transactions_silver
    .filter(~F.col("from").isin(HOUSE_ACCOUNTS))
    .filter(~F.col("to").isin(HOUSE_ACCOUNTS))
    .select(
        F.col("from").alias("src"),
        F.col("to").alias("dst"),
        F.col("amount"),
        F.col("id").alias("transaction_id")
    )
    .dropDuplicates(["src", "dst"])
)

paths = edges.select(
    F.array("src", "dst").alias("accounts"),
    F.array("transaction_id").alias("transaction_ids"),
    F.col("dst").alias("current_account"),
    F.col("amount").alias("total_amount"),
    F.lit(2).alias("account_count")
)

findings = []

# Expand paths until all cycles of up to 6 accounts are checked
for iteration in range(MAX_ACCOUNTS - 1):

    p = paths.alias("p")
    e = edges.alias("e")

    extended = (
        p.join(e, F.col("p.current_account") == F.col("e.src"))
        .filter(
            (
                (F.col("e.dst") == F.element_at(F.col("p.accounts"), 1)) &
                (F.col("p.account_count") >= MIN_ACCOUNTS)
            )
            |
            (
                ~F.array_contains(F.col("p.accounts"), F.col("e.dst")) &
                (F.col("p.account_count") < MAX_ACCOUNTS)
            )
        )
        .select(
            F.col("p.accounts"),
            F.col("p.transaction_ids"),
            F.col("p.total_amount"),
            F.col("p.account_count"),
            F.col("e.dst").alias("next_account"),
            F.col("e.amount").alias("next_amount"),
            F.col("e.transaction_id").alias("next_transaction_id")
        )
    )

    # Capture completed cycles
    completed = (
        extended
        .filter(
            F.col("next_account") == F.element_at("accounts", 1)
        )
        .select(
            F.col("accounts"),
            F.concat(
                "transaction_ids",
                F.array("next_transaction_id")
            ).alias("transaction_ids"),
            (F.col("total_amount") + F.col("next_amount"))
            .alias("total_amount")
        )
    )

    findings.append(completed)

    # Continue extending incomplete paths
    paths = (
        extended
        .filter(
            F.col("next_account") != F.element_at("accounts", 1)
        )
        .select(
            F.concat("accounts", F.array("next_account")).alias("accounts"),
            F.concat(
                "transaction_ids",
                F.array("next_transaction_id")
            ).alias("transaction_ids"),
            F.col("next_account").alias("current_account"),
            (F.col("total_amount") + F.col("next_amount"))
            .alias("total_amount"),
            (F.col("account_count") + 1).alias("account_count")
        )
    )

# Combine and remove rotational duplicates
circular_results = findings[0]

for result in findings[1:]:
    circular_results = circular_results.unionByName(result)

circular_results = (
    circular_results
    .filter(F.element_at("accounts", 1) == F.array_min("accounts"))
    .dropDuplicates(["transaction_ids"])
    .withColumn("pattern", F.lit("circular_transaction"))
    .select("pattern", "accounts", "transaction_ids", "total_amount")
)

display(circular_results)

pattern,accounts,transaction_ids,total_amount
circular_transaction,"List(CIRCLE_A, CIRCLE_B, CIRCLE_C)","List(CIRCLE001, CIRCLE002, CIRCLE003)",14700.0


In [0]:
circular_results.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("circular_transactions_gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Original algorithm parameters
Z_SCORE_THRESHOLD = 3
MIN_HISTORY_DAYS = 7

# Calculate daily transaction counts per account
daily_counts = (
    transactions_silver
    .withColumn("date", F.to_date("timestamp"))
    .groupBy("account", "date")
    .agg(
        F.count("*").alias("transaction_count")
    )
)

# Historical window: all previous observed days, excluding current day
history_window = (
    Window
    .partitionBy("account")
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, -1)
)

# Calculate historical statistics
volume_analysis = (
    daily_counts
    .withColumn(
        "history_days",
        F.count("*").over(history_window)
    )
    .withColumn(
        "historical_average",
        F.avg("transaction_count").over(history_window)
    )
    .withColumn(
        "historical_std_dev",
        F.stddev_samp("transaction_count").over(history_window)
    )
    .filter(F.col("history_days") >= MIN_HISTORY_DAYS)
)

# Calculate Z-score, handling zero standard deviation
volume_analysis = (
    volume_analysis
    .withColumn(
        "z_score",
        F.when(
            F.col("historical_std_dev") == 0,
            F.when(
                F.col("transaction_count") > F.col("historical_average"),
                F.lit(Z_SCORE_THRESHOLD + 1)
            ).otherwise(F.lit(0))
        ).otherwise(
            (
                F.col("transaction_count") -
                F.col("historical_average")
            ) / F.col("historical_std_dev")
        )
    )
)

# Identify unusual transaction volume
unusual_volume_results = (
    volume_analysis
    .filter(F.col("z_score") >= Z_SCORE_THRESHOLD)
    .select(
        F.lit("unusual_transaction_volume").alias("pattern"),
        "account",
        F.col("date").cast("string").alias("date"),
        "transaction_count",
        F.round("historical_average", 2).alias("historical_average"),
        F.round("historical_std_dev", 2).alias("historical_std_dev"),
        F.round("z_score", 2).alias("z_score")
    )
)

display(unusual_volume_results)

pattern,account,date,transaction_count,historical_average,historical_std_dev,z_score
unusual_transaction_volume,VOLUME_A,2026-11-05,20,2.0,0.0,4.0


In [0]:
unusual_volume_results.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("unusual_volume_gold")

In [0]:
from pyspark.sql import functions as F

accounts_silver = (
    spark.table("accounts_bronze")
    .select(
        F.col("account").cast("string"),
        F.col("phone").cast("string"),
        F.col("email").cast("string"),
        F.col("payment_card").cast("string")
    )
)

accounts_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("accounts_silver")

In [0]:

tables = [
    "structuring_alerts_gold",
    "transaction_chains_gold",
    "rapid_cashouts_gold",
    "unusual_volume_gold",
    "circular_transactions_gold"
]

for table in tables:
    print(f"\n========== {table} ==========")
    result = spark.table(table)
    print(f"Findings: {result.count()}")
    display(result)



========== structuring_alerts_gold ==========
Findings: 1


pattern,account,transaction_ids,transaction_count,total_amount,window_minutes
transaction_structuring,STRUCT_A,"List(STRUCT001, STRUCT002, STRUCT003)",3,10500.0,60



========== transaction_chains_gold ==========
Findings: 1


pattern,accounts,transaction_ids,total_amount,elapsed_minutes,amount_similarity,hops
suspicious_transaction_chain,"List(CHAIN_A, CHAIN_B, CHAIN_C, CHAIN_D)","List(CHAIN001, CHAIN002, CHAIN003)",14700.0,8.0,0.96,3



========== rapid_cashouts_gold ==========
Findings: 1


pattern,account,deposit_id,withdrawal_ids,deposit_amount,total_withdrawn,total_wagered,cashout_ratio,wager_ratio,elapsed_minutes
rapid_cashout,CASH_A,CASH001,List(CASH003),9000.0,8900.0,100.0,0.9889,0.0111,15.0



========== unusual_volume_gold ==========
Findings: 1


pattern,account,date,transaction_count,historical_average,historical_std_dev,z_score
unusual_transaction_volume,VOLUME_A,2026-11-05,20,2.0,0.0,4.0



========== circular_transactions_gold ==========
Findings: 1


pattern,accounts,transaction_ids,total_amount
circular_transaction,"List(CIRCLE_A, CIRCLE_B, CIRCLE_C)","List(CIRCLE001, CIRCLE002, CIRCLE003)",14700.0


In [0]:

from pyspark.sql import functions as F

EMPTY_ARRAY = F.array().cast("array<string>")


def normalize_alerts(
    table_name,
    account_column=None,
    transaction_column=None,
    amount_column=None
):
    df = spark.table(table_name)

    # Normalize account information
    if "accounts" in df.columns:
        accounts = F.col("accounts")
    elif account_column and account_column in df.columns:
        accounts = F.array(F.col(account_column).cast("string"))
    else:
        accounts = EMPTY_ARRAY

    # Normalize transaction IDs
    if transaction_column == "rapid":
        transaction_ids = F.concat(
            F.array(F.col("deposit_id").cast("string")),
            F.coalesce(F.col("withdrawal_ids"), EMPTY_ARRAY)
        )
    elif transaction_column and transaction_column in df.columns:
        transaction_ids = F.col(transaction_column)
    else:
        transaction_ids = EMPTY_ARRAY

    # Preserve original detection output as evidence
    evidence = F.to_json(
        F.struct(*[F.col(c) for c in df.columns])
    )

    # Normalize monetary value
    if amount_column and amount_column in df.columns:
        total_amount = F.col(amount_column).cast("double")
    else:
        total_amount = F.lit(None).cast("double")

    return df.select(
        F.col("pattern"),
        accounts.alias("accounts"),
        transaction_ids.alias("transaction_ids"),
        total_amount.alias("total_amount"),
        evidence.alias("evidence_json")
    )


# Normalize all five detection outputs
alert_datasets = [
    normalize_alerts(
        "structuring_alerts_gold",
        account_column="account",
        transaction_column="transaction_ids",
        amount_column="total_amount"
    ),
    normalize_alerts(
        "transaction_chains_gold",
        transaction_column="transaction_ids",
        amount_column="total_amount"
    ),
    normalize_alerts(
        "rapid_cashouts_gold",
        account_column="account",
        transaction_column="rapid",
        amount_column="total_withdrawn"
    ),
    normalize_alerts(
        "unusual_volume_gold",
        account_column="account"
    ),
    normalize_alerts(
        "circular_transactions_gold",
        transaction_column="transaction_ids",
        amount_column="total_amount"
    )
]


# Combine all five detector outputs
unified_alerts = alert_datasets[0]

for dataset in alert_datasets[1:]:
    unified_alerts = unified_alerts.unionByName(dataset)


# Generate stable alert IDs and attach detection thresholds
unified_alerts = (
    unified_alerts
    .withColumn(
        "alert_id",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("pattern"),
                F.to_json(F.col("accounts")),
                F.to_json(F.col("transaction_ids")),
                F.col("evidence_json")
            ),
            256
        )
    )
    .withColumn("detected_at", F.current_timestamp())
    .withColumn(
        "thresholds_json",
        F.when(
            F.col("pattern") == "transaction_structuring",
            F.lit('{"threshold":10000,"window_minutes":60,"min_transactions":3}')
        ).when(
            F.col("pattern") == "suspicious_transaction_chain",
            F.lit('{"min_hops":3,"max_hops":5,"max_time_minutes":15,"amount_similarity":0.8}')
        ).when(
            F.col("pattern") == "rapid_cashout",
            F.lit('{"min_cashout_ratio":0.8,"max_wager_ratio":0.2,"max_time_minutes":60}')
        ).when(
            F.col("pattern") == "unusual_transaction_volume",
            F.lit('{"z_score_threshold":3,"min_history_days":7}')
        ).when(
            F.col("pattern") == "circular_transaction",
            F.lit('{"min_accounts":3,"max_accounts":6}')
        )
    )
    .select(
        "alert_id",
        "pattern",
        "accounts",
        "transaction_ids",
        "total_amount",
        "evidence_json",
        "detected_at",
        "thresholds_json"
    )
)


# Save unified Gold table
unified_alerts.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aml_alerts_gold")


# Display final results
display(spark.table("aml_alerts_gold"))


alert_id,pattern,accounts,transaction_ids,total_amount,evidence_json,detected_at,thresholds_json
bf67c058fcd65adac688cad63463900b9a1806548cdb3341b1865c315674945b,suspicious_transaction_chain,"List(CHAIN_A, CHAIN_B, CHAIN_C, CHAIN_D)","List(CHAIN001, CHAIN002, CHAIN003)",14700.0,"{""pattern"":""suspicious_transaction_chain"",""accounts"":[""CHAIN_A"",""CHAIN_B"",""CHAIN_C"",""CHAIN_D""],""transaction_ids"":[""CHAIN001"",""CHAIN002"",""CHAIN003""],""total_amount"":14700.0,""elapsed_minutes"":8.0,""amount_similarity"":0.96,""hops"":3}",2026-11-07T23:04:56.469Z,"{""min_hops"":3,""max_hops"":5,""max_time_minutes"":15,""amount_similarity"":0.8}"
746ff669f59860e238a985f9cb5a2f79fabdbc0b811afff8ff1d5553a2bc67eb,rapid_cashout,List(CASH_A),"List(CASH001, CASH003)",8900.0,"{""pattern"":""rapid_cashout"",""account"":""CASH_A"",""deposit_id"":""CASH001"",""withdrawal_ids"":[""CASH003""],""deposit_amount"":9000.0,""total_withdrawn"":8900.0,""total_wagered"":100.0,""cashout_ratio"":0.9889,""wager_ratio"":0.0111,""elapsed_minutes"":15.0}",2026-11-07T23:04:56.469Z,"{""min_cashout_ratio"":0.8,""max_wager_ratio"":0.2,""max_time_minutes"":60}"
b6f3cd996e01e6761bdde4ba973afadd84dd96525567b35ecc44f89e6492b3fd,transaction_structuring,List(STRUCT_A),"List(STRUCT001, STRUCT002, STRUCT003)",10500.0,"{""pattern"":""transaction_structuring"",""account"":""STRUCT_A"",""transaction_ids"":[""STRUCT001"",""STRUCT002"",""STRUCT003""],""transaction_count"":3,""total_amount"":10500.0,""window_minutes"":60}",2026-11-07T23:04:56.469Z,"{""threshold"":10000,""window_minutes"":60,""min_transactions"":3}"
01053cb7b0fdb48130332b23031d3f9c00fedcf6242dcc1d24589ac146023af1,unusual_transaction_volume,List(VOLUME_A),List(),null,"{""pattern"":""unusual_transaction_volume"",""account"":""VOLUME_A"",""date"":""2026-11-05"",""transaction_count"":20,""historical_average"":2.0,""historical_std_dev"":0.0,""z_score"":4.0}",2026-11-07T23:04:56.469Z,"{""z_score_threshold"":3,""min_history_days"":7}"
da075a92e7d3ea3444c7291992be8e59e4c3796ed747610c60fad1c4c9c53122,circular_transaction,"List(CIRCLE_A, CIRCLE_B, CIRCLE_C)","List(CIRCLE001, CIRCLE002, CIRCLE003)",14700.0,"{""pattern"":""circular_transaction"",""accounts"":[""CIRCLE_A"",""CIRCLE_B"",""CIRCLE_C""],""transaction_ids"":[""CIRCLE001"",""CIRCLE002"",""CIRCLE003""],""total_amount"":14700.0}",2026-11-07T23:04:56.469Z,"{""min_accounts"":3,""max_accounts"":6}"


In [0]:
display(
    spark.table("aml_alerts_gold")
    .groupBy("pattern")
    .count()
    .orderBy("pattern")
)

pattern,count
circular_transaction,1
rapid_cashout,1
suspicious_transaction_chain,1
transaction_structuring,1
unusual_transaction_volume,1


In [0]:
tables = [
    "structuring_alerts_gold",
    "transaction_chains_gold",
    "rapid_cashouts_gold",
    "unusual_volume_gold",
    "circular_transactions_gold"
]

for table in tables:
    df = spark.table(table)
    print(f"\n===== {table} =====")
    print("Rows:", df.count())
    df.printSchema()


===== structuring_alerts_gold =====
Rows: 1
root
 |-- pattern: string (nullable = true)
 |-- account: string (nullable = true)
 |-- transaction_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- transaction_count: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- window_minutes: integer (nullable = true)


===== transaction_chains_gold =====
Rows: 1
root
 |-- pattern: string (nullable = true)
 |-- accounts: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- transaction_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- total_amount: double (nullable = true)
 |-- elapsed_minutes: double (nullable = true)
 |-- amount_similarity: double (nullable = true)
 |-- hops: integer (nullable = true)


===== rapid_cashouts_gold =====
Rows: 1
root
 |-- pattern: string (nullable = true)
 |-- account: string (nullable = true)
 |-- deposit_id: string (nullable = true)
 |-- withdrawal_ids: arra

In [0]:
display(
    spark.sql("SHOW TABLES IN workspace.default")
)

database,tableName,isTemporary
default,accounts_bronze,false
default,accounts_silver,false
default,aml_alerts_gold,false
default,circular_transactions_gold,false
default,multiple_accounts_gold,false
default,rapid_cashouts_gold,false
default,structuring_alerts_gold,false
default,transaction_chains_gold,false
default,transactions_bronze,false
default,transactions_silver,false


In [0]:
tables = [
    "structuring_alerts_gold",
    "transaction_chains_gold",
    "rapid_cashouts_gold",
    "unusual_volume_gold",
    "circular_transactions_gold"
]

for table in tables:
    df = spark.table(table)
    print(f"\n===== {table} =====")
    print("Rows:", df.count())
    df.printSchema()


===== structuring_alerts_gold =====
Rows: 1
root
 |-- pattern: string (nullable = true)
 |-- account: string (nullable = true)
 |-- transaction_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- transaction_count: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- window_minutes: integer (nullable = true)


===== transaction_chains_gold =====
Rows: 1
root
 |-- pattern: string (nullable = true)
 |-- accounts: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- transaction_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- total_amount: double (nullable = true)
 |-- elapsed_minutes: double (nullable = true)
 |-- amount_similarity: double (nullable = true)
 |-- hops: integer (nullable = true)


===== rapid_cashouts_gold =====
Rows: 1
root
 |-- pattern: string (nullable = true)
 |-- account: string (nullable = true)
 |-- deposit_id: string (nullable = true)
 |-- withdrawal_ids: arra

In [0]:
alerts = spark.table("aml_alerts_gold")

print("Total alerts:", alerts.count())

display(
    alerts.groupBy("pattern").count()
)

Total alerts: 5


pattern,count
suspicious_transaction_chain,1
rapid_cashout,1
transaction_structuring,1
unusual_transaction_volume,1
circular_transaction,1


In [0]:

from pyspark.sql import functions as F

# Check the uploaded source files
transactions_path = "/Volumes/workspace/default/casinoguard_data/transactions.csv"
accounts_path = "/Volumes/workspace/default/casinoguard_data/accounts.csv"

transactions = spark.read.option("header", True).csv(transactions_path)
accounts = spark.read.option("header", True).csv(accounts_path)

print("========== TRANSACTIONS ==========")
print("Rows:", transactions.count())
print("Columns:", transactions.columns)
display(transactions.limit(10))

print("========== ACCOUNTS ==========")
print("Rows:", accounts.count())
print("Columns:", accounts.columns)
display(accounts.limit(10))


========== TRANSACTIONS ==========
Rows: 49
Columns: ['id', 'account', 'from', 'to', 'amount', 'timestamp', 'type']


id,account,from,to,amount,timestamp,type
NORMAL001,PLAYER001,PLAYER001,CASINO,200,2026-11-05T09:00:00,deposit
NORMAL002,PLAYER001,PLAYER001,CASINO,50,2026-11-05T09:10:00,wager
NORMAL003,PLAYER001,CASINO,PLAYER001,100,2026-11-05T09:30:00,withdrawal
CHAIN001,CHAIN_A,CHAIN_A,CHAIN_B,5000,2026-11-05T10:00:00,transfer
CHAIN002,CHAIN_B,CHAIN_B,CHAIN_C,4900,2026-11-05T10:04:00,transfer
CHAIN003,CHAIN_C,CHAIN_C,CHAIN_D,4800,2026-11-05T10:08:00,transfer
STRUCT001,STRUCT_A,STRUCT_A,CAGE001,4000,2026-11-05T11:00:00,deposit
STRUCT002,STRUCT_A,STRUCT_A,CAGE002,3500,2026-11-05T11:15:00,deposit
STRUCT003,STRUCT_A,STRUCT_A,CAGE003,3000,2026-11-05T11:30:00,deposit
CASH001,CASH_A,CASH_A,CASINO,9000,2026-11-05T12:00:00,deposit


========== ACCOUNTS ==========
Rows: 4
Columns: ['account', 'phone', 'email', 'payment_card']


account,phone,email,payment_card
LINK_A,555-1111,linka@example.com,CARD-101
LINK_B,555-1111,linkb@example.com,CARD-101
LINK_C,555-2222,linkc@example.com,CARD-202
NORMAL_ACCOUNT,555-3333,normal@example.com,CARD-303
